# Topic 03 · Checking recent research on generative models and post-training: copying, collapse, forgetting, chain of thought

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/cd03-nghien-cuu-sinh-va-hau-huan-luyen-en.ipynb)

This notebook accompanies [Topic 03 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/nghien-cuu-sinh-va-hau-huan-luyen/), a deep-learning course originally in Vietnamese.
It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/cd03-nghien-cuu-sinh-va-hau-huan-luyen.ipynb)):
the code, the data and every check are the same.

Five recent research results on generative models and on continuing to train an existing model, each checked again on
a problem small enough to run on a CPU. As in Topic 02, we rebuild each experiment, run several seeds, and state
clearly whether the result **holds**, **does not hold** or **depends**.

**What you'll do:**
1. Build a "perfect diffusion model" from a formula, see that it can only copy, then add locality and see it create
   new digits (Kamb & Ganguli, 2025).
2. Measure when a real diffusion model starts copying, as a function of the number of training images (Bonnaire et
   al., 2025).
3. Train many generations of models on data generated by the previous generation: replacing the data collapses,
   keeping it doesn't (Shumailov et al., 2024; Gerstgrasser et al., 2024).
4. Compare SFT and reinforcement learning for continuing to train a model: which one forgets more, and why (Shenfeld
   et al., 2025).
5. Teach a small transformer to compute the parity of a few secret bits, with and without chain of thought (Kim &
   Suzuki, 2025).

**Background:** [Lesson 21 · Diffusion & flow matching](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/diffusion/),
[Lesson 22 · Pretraining & fine-tuning](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/tien-huan-luyen/),
[Lesson 26 · From language model to assistant](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/hau-huan-luyen/) and
[Lesson 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/transformer/).
**Run time:** about 1 minute on a CPU, including the first data download.

The notebook uses **PyTorch**. Each section runs a short version live, then loads the full results (several seeds,
longer runs) that the course's scripts `reel/research/*.py` computed in advance, with the same code and the same seeds.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

In [ ]:
import gzip
import hashlib
import json
import math
import os
import re
import urllib.request
from itertools import pairwise
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "2")
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter
from torch import nn

torch.set_num_threads(2)

BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("orange_gray_blue", [ORANGE, "#f2f2f2", BLUE])  # negative → 0 → positive
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number with thousands separators: 13,007 · 0.1 · −4.38 (never “−0”)."""
    return f"{round(float(x), digits) + 0.0:,.{digits}f}".replace("-", "−")


def pct(p, digits=1):
    return num(100 * p, digits) + "%"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def tick(v, _pos=None):
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    if 0.01 <= v <= 1000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Axis labels in the notebook's number format; call right before plt.show(). Short log axes get 1–2–5 ticks,
    long ones powers of 10."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_locator(NullLocator())
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))

## 2 · Data

MNIST and Fashion-MNIST (all four files of each, checked by MD5), *The Tale of Kiều* (*Truyện Kiều*, Nguyễn Du's
Vietnamese poem) from Wikisource (pinned at revision 205132, as in Lessons 15 and 17), and the five results files of the
scripts.

In [ ]:
MNIST_URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
MNIST_FILES = {
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
FASHION_URL = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/"
FASHION_FILES = {
    "train-images-idx3-ubyte.gz": "8d4fb7e6c68d591d4c3dfef9ec88bf0d",
    "train-labels-idx1-ubyte.gz": "25c81989df183df01b3e8a0aad5dffbe",
    "t10k-images-idx3-ubyte.gz": "bef4ecab320f06d8554ea6380940ec79",
    "t10k-labels-idx1-ubyte.gz": "bb300cfdad3c16e7a12a480ee83cd310",
}
KIEU_SRC = "https://vi.wikisource.org/w/index.php?oldid=205132&action=raw"
KIEU_RAW = Path("truyen_kieu_205132.wiki")
opener = urllib.request.build_opener()
opener.addheaders = [("User-Agent", "HocSauReels-notebook/1.0 (education; https://d3qr1cjln6qqtk.cloudfront.net/)")]
urllib.request.install_opener(opener)  # Wikimedia rejects requests without a User-Agent
for folder, url, files in (("mnist", MNIST_URL, MNIST_FILES), ("fashion", FASHION_URL, FASHION_FILES)):
    Path(folder).mkdir(exist_ok=True)
    for name, md5 in files.items():
        p = Path(folder) / name
        if not p.exists():
            urllib.request.urlretrieve(url + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{p} is corrupted; delete it and download it again"
if not KIEU_RAW.exists():
    urllib.request.urlretrieve(KIEU_SRC, KIEU_RAW)
assert hashlib.md5(KIEU_RAW.read_bytes()).hexdigest() == "d4090daac094841888ee2b2b18661065"


def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


LOCAL = fetch_json("research-diffusion-local.json", "64522a8076946d83c1d1b3ca78dc7ce1")
MEMO = fetch_json("research-memo.json", "948f96f67edb5ffd8534b9a41a9ebed6")
COLLAPSE = fetch_json("research-collapse.json", "0202c9477632ae5b0692ca67ff8225fe")
RAZOR = fetch_json("research-razor.json", "f5945b63dda5864e63e6d44b9c868479")
COT = fetch_json("research-cot.json", "42729fb56fdd9d6c96e7304f4a016956")

In [ ]:
def idx(folder, kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(Path(folder) / f"{kind}-images-idx3-ubyte.gz") as f:
        x = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(Path(folder) / f"{kind}-labels-idx1-ubyte.gz") as f:
        y = np.frombuffer(f.read(), np.uint8, offset=8)
    return x, y.astype(np.int64)


mnist_tr, mnist_ytr = idx("mnist", "train")
mnist_te, mnist_yte = idx("mnist", "t10k")
fash_tr, fash_ytr = idx("fashion", "train")
fash_te, fash_yte = idx("fashion", "t10k")
print("MNIST", mnist_tr.shape, mnist_te.shape, "· Fashion-MNIST", fash_tr.shape, fash_te.shape)

## 3 · Diffusion models: copying or creating? A closed-form solution

**Claim** (Kamb & Ganguli, ICML 2025). Lesson 21 already met the **exact denoiser**: the exact solution of the
denoising problem on a finite training set. It is a softmax-weighted average of the training images, and sampling with
it can only ever return exactly one training image. A diffusion model that learned its objective *perfectly* would only
copy. So why do real models produce new images? Kamb & Ganguli show that it is enough to add two constraints that
convolutional networks have built in: **locality** (each output pixel only looks at a $P \times P$ patch around it)
and, optionally, **translation equivariance**. The solution can then still be written in closed form, and it produces new
images like **mosaics** pieced together from patches of the training images. They also find that these closed-form
"machines" predict the outputs of trained U-Nets quite closely. A later paper (Lukoianov et al., 2025) counters that
locality doesn't need the network: it is already present in the correlations between the pixels of the data.

The experiment: 1,000 MNIST training images, pixels scaled to $[-1, 1]$; the noise path as in Lesson 21,
$x_t = (1 - t)\,x_0 + t\,\varepsilon$, with $t = 1$ being pure noise. Nothing is trained:
- **ideal:** $\hat x_0(x_t) = \sum_i w_i x_i$ with $w_i \propto \exp\!\big(-\lVert x_t - (1-t)x_i\rVert^2 / 2t^2\big)$
  over the **whole image**;
- **local:** the same softmax, but computed separately for each pixel over the $P \times P$ patch around it, so each
  pixel can "follow" a different training image.

Sampling takes 20 deterministic steps from $t = 1$ down to 0, as in DDIM. A sample counts as a **copy** if it is more
than 3 times closer to its nearest training image than to the second nearest.

In [ ]:
TRAIN = mnist_tr[:1000].astype(np.float64) / 127.5 - 1
TIMES = np.linspace(1, 0, 21) ** 1.5  # 20 steps, denser near t = 0, where the details are decided


def softmax_w(D, t):
    """Softmax along axis 0 (the training images) of −D / 2t²; the weights sum to 1 at every pixel."""
    L = -D / (2 * t * t)
    w = np.exp(L - L.max(0, keepdims=True))
    return w / w.sum(0, keepdims=True)


def ideal(xt, t):
    D = ((xt[None] - (1 - t) * TRAIN) ** 2).sum((1, 2))  # one distance per training image
    return np.tensordot(softmax_w(D, t), TRAIN, 1)


def boxsum(a, P):
    """Sum of a over the P×P window around each pixel (clipped at the edges), via 2-D cumulative sums. a: (n, 28, 28)."""
    r = P // 2
    c = np.pad(a, ((0, 0), (1, 0), (1, 0))).cumsum(1).cumsum(2)
    lo, hi = np.clip(np.arange(28) - r, 0, 28), np.clip(np.arange(28) + r + 1, 0, 28)
    return c[:, hi][:, :, hi] - c[:, lo][:, :, hi] - c[:, hi][:, :, lo] + c[:, lo][:, :, lo]


def local(xt, t, P):
    w = softmax_w(boxsum((xt[None] - (1 - t) * TRAIN) ** 2, P), t)  # one distance per (image, pixel)
    return (w * TRAIN).sum(0)


def sample(den, seed):
    """From pure noise to an image: at each step, guess x̂₀, infer ε from it, then move x to the next time."""
    x = np.random.default_rng(seed).standard_normal((28, 28))
    for t, s in pairwise(TIMES):
        x0 = den(x, t)
        eps = (x - (1 - t) * x0) / t
        x = (1 - s) * x0 + s * eps
    return x


def nearest(x):
    """Distance to the nearest training image, the nearest / second-nearest ratio, and the nearest image's index."""
    d = np.sqrt(((TRAIN - x) ** 2).sum((1, 2)))
    i = np.argsort(d)
    return d[i[0]], d[i[0]] / d[i[1]], int(i[0])

### 🤔 Predict first
Sample from the ideal denoiser and from the local denoiser with $P = 9$, starting from the same noise. What percentage
of copies does each one give?

In [ ]:
live = {"ideal": [sample(ideal, s) for s in (0, 1)],
        "local, P = 9": [sample(lambda z, t: local(z, t, 9), s) for s in (0, 1)]}  # fmt: skip
fig, axes = plt.subplots(2, 4, figsize=(7, 3.8))
for row, (name, xs) in enumerate(live.items()):
    for j, x in enumerate(xs):
        d, ratio, i = nearest(x)
        axes[row, 2 * j].imshow(x, cmap="gray", vmin=-1, vmax=1)
        axes[row, 2 * j].set_title(f"{name}\nsample {j + 1}", fontsize=8)
        axes[row, 2 * j + 1].imshow(TRAIN[i], cmap="gray", vmin=-1, vmax=1)
        axes[row, 2 * j + 1].set_title(f"nearest training image\ndistance {num(d, 1)}", fontsize=8)
        print(f"{name:14} sample {j + 1}: nearest {num(d, 2)}, ratio {num(ratio, 3)} → "
              f"{'COPY' if ratio < 1 / 3 else 'new image'}")  # fmt: skip
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()
for j in range(2):
    assert nearest(live["ideal"][j])[2] == LOCAL["ideal"]["nnIndex"][j], "differs from the saved results"
    assert abs(nearest(live["local, P = 9"][j])[0] / LOCAL["local"]["9"]["nn"][j] - 1) < 0.02, (
        "differs from the saved results"
    )

Top row: the ideal denoiser returns **exactly** a training image (a distance of 0 to the image beside it). Bottom row:
the same starting noise, but each pixel only looks at the 9 × 9 patch around it; the result is new handwriting that
matches no training image. The full results, 16 samples for each patch size:

In [ ]:
print(f"{'':20} copies   distance to the nearest training image (median)")
print(f"{'real test images':20} {pct(LOCAL['test']['copy'], 0):>6}   {num(np.median(LOCAL['test']['nn']), 2)}")
print(f"{'ideal':20} {pct(LOCAL['ideal']['copy'], 0):>6}   {num(np.median(LOCAL['ideal']['nn']), 2)}")
for P in LOCAL["patches"]:
    r_ = LOCAL["local"][str(P)]
    print(f"{f'local, P = {P}':20} {pct(r_['copy'], 0):>6}   {num(np.median(r_['nn']), 2)}")
print(f"{'local + translation, P = 5 (256 images)':20} {pct(LOCAL['els5']['copy'], 0):>6}   "
      f"{num(np.median(LOCAL['els5']['nn']), 2)}")  # fmt: skip

rows = [("ideal", LOCAL["ideal"]["img"])] + [(f"P = {P}", LOCAL["local"][str(P)]["img"]) for P in (3, 5, 9, 13)]
rows.append(("translation, P = 5", LOCAL["els5"]["img"]))
fig, axes = plt.subplots(len(rows), 8, figsize=(8, 1.1 * len(rows)))
for i, (name, imgs) in enumerate(rows):
    for j in range(8):
        axes[i, j].imshow(np.array(imgs[j], np.uint8).reshape(28, 28), cmap="gray", vmin=0, vmax=255)
        axes[i, j].axis("off")
    axes[i, 0].set_title(name, fontsize=8, loc="left")
plt.tight_layout()
plt.show()

Each row is 8 samples from one denoiser, from the same 8 starting noises. The top row (ideal) is all verbatim copies
of training images. Small patches ($P = 3$) give scattered fragments: each pixel picks a different image on its own,
and nothing holds the strokes together. Medium patches ($P = 5$ to 9) give new, connected strokes, sometimes a cross
between two digits. Large patches ($P = 13$) come closer to the training images again (6% are copies): the larger the
patch, the closer the denoiser gets to the ideal one. The last row adds translation equivariance (patches taken from
**any position** of any image, over 256 images): with a fixed $P = 5$, the result is disconnected strokes that don't
form digits.

And Lukoianov et al.'s critique: is the optimal **linear** denoiser (which uses only the mean and covariance of the
data, no network at all) already local? The cell below plots its weights for the pixel at the center of the image, at
$t = 0.5$.

In [ ]:
lin = LOCAL["linear"]
row = np.array(lin["row"]).reshape(28, 28)
fig, ax = plt.subplots(figsize=(3.6, 3.2))
lim = np.abs(row).max()
im = ax.imshow(row, cmap=DIVERGING, vmin=-lim, vmax=lim)
ax.plot(*lin["pixel"][::-1], marker="+", color="black", ms=10)
ax.set_title(f"weights for pixel {tuple(lin['pixel'])}, t = {num(lin['t'], 1)}", fontsize=9)
ax.axis("off")
fig.colorbar(im, ax=ax, fraction=0.046)
fmt_axes(fig)
plt.show()
for rad, share in zip(lin["radii"], lin["massWithin"]):
    print(f"within {rad} pixel{'' if rad == 1 else 's'}: {pct(share, 0)} of the total magnitude of the weights")

The heat map shows the weights of the pixel marked "+" (blue: positive, orange: negative). They cluster around that
very pixel: almost half of the total magnitude lies within a radius of 3 pixels, even though the denoiser is allowed to
look at the whole image. So the data by itself already pushes the optimal solution toward locality, just as Lukoianov
et al. say; that doesn't refute Kamb & Ganguli (locality is still what turns copying into creating), it only suggests
that locality doesn't necessarily come from the network's architecture.

**Conclusion: the main part holds; the rest depends.** The exact denoiser copies 100% of the time; restricting it to
local patches ends the copying (0% for $P \le 9$) and produces new handwriting for $P$ around 5–9. But the quality
depends strongly on the patch size, and we use **one** patch size for every $t$; Kamb & Ganguli choose a patch size
that changes with $t$, fitted to the trained network. We don't check their main quantitative result: that the
closed-form machines predict the U-Net's outputs.

## 4 · When does a diffusion model start copying?

**Claim** (Bonnaire, Urfin, Biroli & Mézard, NeurIPS 2025): training has **two timescales**. The model starts
generating good, *new* samples from a time $\tau_{\text{gen}}$ that hardly depends on the number of training images
$n$. It only starts copying training images later, at $\tau_{\text{mem}}$, and **$\tau_{\text{mem}}$ grows in
proportion to $n$**. Between the two lies a window in which the model generalizes, and that window widens as $n$ grows.

The experiment, Lesson 21's latent diffusion in miniature: the data are the 16-dimensional codes of MNIST images from
Lesson 19's autoencoder; the model is a residual MLP, width 256, trained with flow matching; $n$ from 32 to 1,024
codes, 3 seeds, 40,000 steps. This section only reads the results (each run takes a few minutes, and needs Lesson 19's
encoder). At each checkpoint, the model generates 1,000 samples, and we measure:
- **the copy rate:** the share of samples closer to their nearest training code than 99% of the real test codes are
  (Lesson 21's rule);
- **FD:** the Fréchet distance between the Gaussians fitted to the samples and to 1,000 test codes (smaller means more
  alike; the cell below prints the distance between two sets of real codes, as a floor).

### 🤔 Predict first
With 32 training codes, the model starts copying (10% of the samples are copies) after 500 steps. With 1,024 codes (32
times as many), after how many steps would you guess?

In [ ]:
def steps_text(es):
    """[500, 500, None] → "500, 500, never"."""
    return ", ".join("never" if e is None else num(e, 0) for e in es)


checks = np.array(MEMO["check"])
print(f"FD between two sets of 1,000 real codes (the floor): {num(MEMO['fdRealVsReal'], 2)}")
onset = {}
for n in MEMO["sizes"]:
    rs = MEMO["runs"][str(n)]
    onset[n] = [next((int(st) for st, c in zip(checks, r_["copy"]) if c >= 0.1), None) for r_ in rs]
    fd = np.median([r_["fd"] for r_ in rs], 0)
    final = [r_["copy"][-1] for r_ in rs]
    print(f"n = {num(n, 0):>5}: ≥ 10% copies from step {steps_text(onset[n])} · {pct(min(final), 0)}–{pct(max(final), 0)} copies at step "
          f"{num(checks[-1], 0)} · FD at step {num(1000, 0)} / {num(checks[-1], 0)}: {num(fd[1], 2)} / {num(fd[-1], 2)}")  # fmt: skip

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
cmap = plt.get_cmap("viridis")
for k, n in enumerate(MEMO["sizes"]):
    rs = MEMO["runs"][str(n)]
    col = cmap(k / (len(MEMO["sizes"]) - 1))
    ls = "-" if k % 2 == 0 else "--"
    axes[0].plot(checks, np.median([r_["copy"] for r_ in rs], 0), color=col, ls=ls, lw=1.8, label=f"n = {num(n, 0)}")
    axes[1].plot(checks, np.median([r_["fd"] for r_ in rs], 0), color=col, ls=ls, lw=1.8, label=f"n = {num(n, 0)}")
ns = np.array(MEMO["sizes"])
med_onset = np.array([np.median(onset[n]) for n in ns])
axes[2].plot(ns, med_onset, "o", color=BLUE, label="copying starts (median of 3 seeds)")
axes[2].plot(ns, med_onset[0] * ns / ns[0], color=GREY, ls=":", label="proportional to n")
axes[0].set(xscale="log", xlabel="step", ylabel="copy rate (median)", title="Copying")
axes[1].set(xscale="log", xlabel="step", ylabel="FD to the test codes (median)", title="Sample quality")
axes[2].set(xscale="log", yscale="log", xlabel="number of training codes n", ylabel="step copying starts",
            title="τ_mem vs. n")  # fmt: skip
axes[0].yaxis.set_major_formatter(FuncFormatter(lambda v, _: pct(v, 0)))
axes[0].legend(frameon=False, fontsize=7)
axes[2].legend(frameon=False, fontsize=7)
fmt_axes(fig)
plt.tight_layout()
plt.show()

Left: the copy rate against the number of steps (log scale), one color per $n$, from purple ($n = 32$) to yellow
($n = 1{,}024$), solid and dashed lines alternating. The larger $n$, the further right its curve: the later the model
copies. Middle: FD against the number of steps. For $n \ge 128$, FD is already close to its best level from about step
1,000 on, and then changes only a little (rising slightly once the model starts copying), while the copy rate keeps
rising: quality comes early, copying comes late. Right: the step at which copying starts, against $n$ (both axes on a
log scale), with the line of direct proportion (gray dots). The points stay close to that line: each time $n$
doubles, the model starts copying about twice as late (more than that the last time), from 500 steps ($n = 32$) to
30,000–40,000 steps ($n = 1{,}024$).

**Conclusion: holds.** The onset of copying comes later in proportion to the amount of training data, while sample
quality is reached early and stays almost unchanged. Limitations: the checkpoints are sparse (500, 1,000, 2,000, …),
so the onset is only accurate to the nearest checkpoint; FD is measured in the 16-dimensional code space with 1,000
samples, so it can only be compared within this experiment. With $n = 1{,}024$, the model hasn't finished copying
after 40,000 steps (12–14%, still rising): with enough data relative to the model's capacity, $\tau_{\text{mem}}$
exceeds any practical training time, and the model never gets around to copying, as the paper predicts.

## 5 · Model collapse: replace the data or keep it?

**Claim.** Shumailov et al. (2024, *Nature*): if each new generation of models learns only from data generated by the
previous generation, the tails of the distribution gradually disappear and the model degenerates: **model collapse**.
Gerstgrasser et al. (2024) reply: that happens when synthetic data **replaces** the real data; if each generation
**adds** its synthetic data to the existing pool (real data included), the error stays bounded.

The simplest experiment: a 1-dimensional normal distribution. Generation 0 estimates the mean and standard deviation
from 100 real samples of $\mathcal{N}(0, 1)$. Each later generation draws 100 samples from the current estimate and
estimates again, in one of two ways:
- **replace:** use only the 100 new samples;
- **accumulate:** use all the samples so far (the 100 real samples plus 100 samples from each generation).

### 🤔 Predict first
Each re-estimation is only slightly off. After 200 generations of replacing, will the estimated standard deviation be
around 1, well below 1, or well above 1?

In [ ]:
def gauss_generations(mode, rng, generations=200, n=100, seeds=200, ddof=0):
    sig = np.empty((seeds, generations + 1))
    for s in range(seeds):
        pool = rng.standard_normal(n)
        mu, sd = pool.mean(), pool.std(ddof=ddof)
        sig[s, 0] = sd
        for g in range(1, generations + 1):
            new = rng.normal(mu, sd, n)
            pool = new if mode == "replace" else np.concatenate([pool, new])
            mu, sd = pool.mean(), pool.std(ddof=ddof)
            sig[s, g] = sd
    return sig


rng = np.random.default_rng(0)  # one random stream for both ways, in this order (as in the script)
gauss = {m: gauss_generations(m, rng) for m in ("replace", "accumulate")}
for m, label in (("replace", "replace"), ("accumulate", "accumulate")):
    med = np.median(gauss[m], 0)
    print(f"{label:10}: median σ at generation 0 / 10 / 50 / 200 = " + " / ".join(num(med[g], 3) for g in (0, 10, 50, 200)) +
          f" · share of seeds with σ < 0.1 at generation 200: {pct((gauss[m][:, 200] < 0.1).mean(), 0)}")  # fmt: skip
    stored = COLLAPSE["gauss"][m]["sigma"][1]
    assert all(abs(med[g] - stored[g]) < 1e-3 for g in (0, 10, 50, 200)), "differs from the saved results"
unbiased = gauss_generations("replace", np.random.default_rng(1), ddof=1)  # divide by n − 1: unbiased variance
med = np.median(unbiased, 0)
print("replace, dividing by n − 1: median σ at generation 0 / 10 / 50 / 200 = " + " / ".join(num(med[g], 3) for g in (0, 10, 50, 200)) +
      f" · share of seeds with σ < 0.1 at generation 200: {pct((unbiased[:, 200] < 0.1).mean(), 0)}")  # fmt: skip
assert np.median(gauss["replace"], 0)[200] < med[200] < 0.5, "dividing by n − 1 must shrink more slowly, yet shrink"

Replace: the standard deviation shrinks steadily, to 0.62 at generation 50 and 0.14 at generation 200; in 40% of the
seeds it is below 0.1. Each step is only slightly off, but **the errors accumulate**, and in the next generation there
is no real data left to pull it back. The error has two parts. One is a small systematic bias: `pool.std()` divides by
$n$, so on average $\hat\sigma^2$ is only $(n - 1)/n$ times $\sigma^2$, and the sample standard deviation comes out
slightly below the true σ each time. The other is random error: each generation multiplies σ by a random factor around
1, and a decrease by some fraction is not fully made up for by an increase by the same fraction
($(1 - a)(1 + a) < 1$), so the product of the factors drifts downward. The last line of the cell above removes the
first part (dividing by $n - 1$, the unbiased variance estimate): σ still shrinks, more slowly, to 0.34 at generation
200. Accumulate: the standard deviation stays put at 0.98, because the 100 real samples are
always there and the new samples are only additions.

A second experiment, closer to language: a character bigram model of *The Tale of Kiều* (the probability of the next
character given the current one, smoothed so that no character has probability 0). Each generation produces about 96
thousand characters, the length of the real text, and then counts again. We measure the bits per character on the last
254 lines of verse (not used for training), and the share of the test text's character pairs that the model still
"knows" (has counted at least once).

In [ ]:
body = KIEU_RAW.read_bytes().decode("utf-8").split("<poem>")[1].split("</poem>")[0]
lines = [re.sub(r"\{\{số\|\d+\}\}", "", ln).strip() for ln in body.split("\n")]
text = "\n".join(ln for ln in lines if ln) + "\n"
assert hashlib.md5(text.encode()).hexdigest() == "dd94ee390bfc543287b11af62b2f2fc8"
CHARS = sorted(set(text))
V = len(CHARS)
stoi = {ch: i for i, ch in enumerate(CHARS)}
ids = np.array([stoi[ch] for ch in text])
cut = len("\n".join(text[:-1].split("\n")[:3000])) + 1
kieu_tr, kieu_te = ids[:cut], ids[cut:]
test_pairs = list(set(zip(kieu_te[:-1].tolist(), kieu_te[1:].tolist())))
ALPHA = 0.01


def bigram_counts(seq):
    C = np.zeros((V, V))
    np.add.at(C, (seq[:-1], seq[1:]), 1)
    return C


def bigram_probs(C):
    P = C + ALPHA
    return P / P.sum(1, keepdims=True)


def bigram_sample(P, n, rng, start):
    cum = P.cumsum(1)
    out = np.empty(n, np.int64)
    c, u = start, rng.random(n)
    for i in range(n):
        c = min(int(np.searchsorted(cum[c], u[i])), V - 1)
        out[i] = c
    return out


def bigram_stats(C):
    bits = -np.log2(bigram_probs(C)[kieu_te[:-1], kieu_te[1:]]).mean()
    seen = np.mean([C[a, b] > 0 for a, b in test_pairs])
    return bits, seen


def bigram_generations(mode, generations=4, seed=0):
    rng = np.random.default_rng(100 + seed)
    C = bigram_counts(kieu_tr)
    out = [bigram_stats(C)]
    for _ in range(generations):
        new = bigram_sample(bigram_probs(C), len(kieu_tr), rng, start=int(kieu_tr[0]))
        C = bigram_counts(new) if mode == "replace" else C + bigram_counts(new)
        out.append(bigram_stats(C))
    return out


print(f"{V} distinct characters; {num(len(kieu_tr), 0)} training characters, {num(len(kieu_te), 0)} test characters")
for m, label in (("replace", "replace"), ("accumulate", "accumulate")):
    st = bigram_generations(m)
    print(f"{label:10}: bits/character, generation 0 → 4: {num(st[0][0], 3)} → {num(st[-1][0], 3)} · "
          f"test character pairs still known: {pct(st[0][1])} → {pct(st[-1][1])}")  # fmt: skip
    assert abs(st[-1][0] - COLLAPSE["bigram"][m]["bits"][0][4]) < 1e-3, "differs from the saved results"

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
gens = np.arange(COLLAPSE["gauss"]["generations"] + 1)
for m, col, ls, label in (("replace", ORANGE, "--", "replace"), ("accumulate", BLUE, "-", "accumulate")):
    q25, q50, q75 = (np.array(v) for v in COLLAPSE["gauss"][m]["sigma"])
    axes[0].plot(gens, q50, color=col, ls=ls, lw=1.8, label=label)
    axes[0].fill_between(gens, q25, q75, color=col, alpha=0.15)
    bits = np.array(COLLAPSE["bigram"][m]["bits"])
    g2 = np.arange(bits.shape[1])
    axes[1].plot(g2, np.median(bits, 0), color=col, ls=ls, lw=1.8, marker="o" if m == "replace" else "s", ms=4,
                 label=label)  # fmt: skip
for m, label in (
    ("replace", "replace"),
    ("accumulate", "accumulate"),
    ("accumulate-fixed", "accumulate, trimmed to the original size"),
):
    b = np.median(np.array(COLLAPSE["bigram"][m]["bits"]), 0)
    seen = np.median(np.array(COLLAPSE["bigram"][m]["testSeen"]), 0)
    print(
        f"{label:40}: generation 15 (median of 5 seeds): {num(b[-1], 3)} bits/character, test pairs still known {pct(seen[-1])}"
    )
bits_fixed = np.array(COLLAPSE["bigram"]["accumulate-fixed"]["bits"])
axes[1].plot(np.arange(bits_fixed.shape[1]), np.median(bits_fixed, 0), color=GREEN, ls="-.", lw=1.8, marker="^", ms=4,
             label="accumulate, trimmed to the original size")  # fmt: skip
axes[0].set(xlabel="generation", ylabel="estimated σ (median, 25–75%)", title="Normal distribution, n = 100")
axes[1].set(xlabel="generation", ylabel="bits/character on the test lines", title="Bigram model of The Tale of Kiều")
axes[0].legend(frameon=False)
axes[1].legend(frameon=False, fontsize=8)
fmt_axes(fig)
plt.tight_layout()
plt.show()

Left: the estimated standard deviation over 200 generations (the line is the median of 200 seeds, the shaded band the
25–75% range). Replace (orange, dashed) drifts toward 0 and its band widens; accumulate (blue, solid) stays put.
Right: the bits per character on the test text over 15 generations, 5 seeds (the numbers are printed above). Replace
gets steadily worse, and the share of test character pairs the model still knows keeps falling: the rare pairs
disappear first. Accumulate barely moves. The green line is Gerstgrasser's test: it still keeps every generation, but
trims the data pool back to exactly its original size, to separate "more data" from "keeping the old data". It is
slightly worse than accumulating but still stable: what prevents the collapse is that **the real data is not thrown
away**, not just having more data.

**Conclusion: both hold.** Replacing the data collapses (Shumailov), keeping the old data doesn't (Gerstgrasser): the
two papers don't contradict each other, they describe two different procedures. One detail of this experiment: no
character disappears completely, because the smoothing always leaves a small probability; what is lost is the rare
character **pairs**.

## 6 · RL's Razor: why reinforcement learning forgets less than SFT

**Claim** (Shenfeld, Pari & Agrawal, 2025): when you continue training a model on a new task, reinforcement learning
(RL) and SFT can reach the same accuracy on the new task, but RL forgets **less** of what the model already knew. They
argue that the forgetting is predicted by a single number: the KL divergence $\mathrm{KL}(\pi_0 \,\|\, \pi)$ between
the original model and the new one, measured on the inputs of the new task. And SFT on the labeling closest to
$\pi_0$ ("optimal SFT") forgets even less than RL. This is a single paper, not yet checked independently by anyone.

We rebuild the paper's own small experiment, at a smaller size. An MLP 785 → 256 → 128 → 10, with a **flag** at the
end of the input (+1: the parity task, −1: Fashion-MNIST):
- **pretraining** on both together: 500 digit images labeled "ParityMNIST"-style (each image gets a fixed random label
  **with the same parity** as the digit) and 500 Fashion-MNIST images with their true labels;
- **the new task**: parity again, but now every label with the right parity is correct, so each image has **5 correct
  answers**. This is exactly what makes RL and SFT differ: SFT has to pick one labeling, RL gets to choose its own.

Three ways to continue training, all on 500 new digit images:
- **SFT:** even → label 0, odd → label 1;
- **RL:** sample 8 labels per image, reward 1 if the parity is right, REINFORCE with a within-group baseline
  (GRPO-style, Lesson 26);
- **optimal SFT:** soft labels $q = \pi_0$ with only the 5 correct labels kept, then renormalized, i.e. the correct
  distribution closest to $\pi_0$.

In [ ]:
def flagged(x, v):
    x = torch.from_numpy(x.reshape(len(x), -1).astype(np.float32) / 255)
    return torch.cat([x, torch.full((len(x), 1), float(v))], 1)


PARITY = torch.tensor([[1.0 if (j % 2) == (d % 2) else 0.0 for j in range(10)] for d in range(10)])
Tpar, ypar = flagged(mnist_te[:2000], 1), torch.from_numpy(mnist_yte[:2000])
Tfash, yfash = flagged(fash_te[:2000], -1), torch.from_numpy(fash_yte[:2000])
Xm_all, ym_all = torch.from_numpy(mnist_tr.reshape(-1, 784).astype(np.float32) / 255), torch.from_numpy(mnist_ytr)
Xf_all, yf_all = torch.from_numpy(fash_tr.reshape(-1, 784).astype(np.float32) / 255), torch.from_numpy(fash_ytr)


def num_lr(v):
    return f"{num(v * 10 ** -math.floor(math.log10(v)), 0)}·10{str(math.floor(math.log10(v))).translate(SUP)}"


def razor_net(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(785, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))


def flag(x, v):
    return torch.cat([x, torch.full((len(x), 1), float(v))], 1)


def pretrain(seed=0, steps=500, n=500):
    """Exactly as in reel/research/razor.py: the same order of random draws."""
    g = torch.Generator().manual_seed(seed)
    im = torch.randperm(len(Xm_all), generator=g)[:n]
    jf = torch.randperm(len(Xf_all), generator=g)[:n]
    lab_par = (ym_all[im] % 2) + 2 * torch.randint(0, 5, (n,), generator=g)
    X = torch.cat([flag(Xm_all[im], 1), flag(Xf_all[jf], -1)])
    Y = torch.cat([lab_par, yf_all[jf]])
    m = razor_net(seed)
    opt = torch.optim.Adam(m.parameters(), 1e-3)
    for _ in range(steps):
        b = torch.randint(0, len(X), (128,), generator=g)
        loss = nn.functional.cross_entropy(m(X[b]), Y[b])
        opt.zero_grad()
        loss.backward()
        opt.step()
    return m, set(im.tolist())


def measure(m, p0_par):
    with torch.no_grad():
        lp = torch.log_softmax(m(Tpar), 1)
        new = (lp.exp() * PARITY[ypar]).sum(1).mean().item()  # probability put on the labels with the right parity
        kl = (p0_par * (torch.log(p0_par + 1e-30) - lp)).sum(1).mean().item()
        old = (m(Tfash).argmax(1) == yfash).float().mean().item()
    return new, old, max(kl, 0.0)


def finetune(method, base_state, used, seed=0, steps=200, lr=1e-3, n=500):
    m = razor_net(seed)
    m.load_state_dict(base_state)
    g = torch.Generator().manual_seed(100 + seed)
    pool = torch.tensor([i for i in torch.randperm(len(Xm_all), generator=g).tolist() if i not in used][:n])
    X, digit = flag(Xm_all[pool], 1), ym_all[pool]
    with torch.no_grad():
        p0_par, p0_train = torch.softmax(m(Tpar), 1), torch.softmax(m(X), 1)
    if method == "sft":
        target = nn.functional.one_hot(digit % 2, 10).float()
    elif method == "oracle":
        q = p0_train * PARITY[digit]
        target = q / q.sum(1, keepdim=True)
    before = measure(m, p0_par)
    opt = torch.optim.Adam(m.parameters(), lr)
    for _ in range(steps):
        b = torch.randint(0, len(X), (64,), generator=g)
        logits = m(X[b])
        if method == "rl":
            lp = torch.log_softmax(logits, 1)
            with torch.no_grad():
                a = torch.multinomial(lp.exp(), 8, replacement=True, generator=g)
                rew = PARITY[digit[b]].gather(1, a)
                adv = rew - rew.mean(1, keepdim=True)
            loss = -(adv * lp.gather(1, a)).mean()
        else:
            loss = -(target[b] * torch.log_softmax(logits, 1)).sum(1).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    return before, measure(m, p0_par)


base_model, used = pretrain()
base_state = {k: v.clone() for k, v in base_model.state_dict().items()}
print("base model pretrained (500 steps)")

### 🤔 Predict first
After 200 steps of further training, all three methods reach about the same accuracy on the parity task. Which one
keeps Fashion-MNIST best, and which one forgets the most?

In [ ]:
names = {"sft": "SFT (even → 0, odd → 1)", "rl": "RL (GRPO-style)", "oracle": "optimal SFT"}
for mth in ("sft", "rl", "oracle"):
    (n0, o0, _), (n1, o1, kl) = finetune(mth, base_state, used)
    print(f"{names[mth]:23} parity {pct(n0)} → {pct(n1)} · Fashion-MNIST {pct(o0)} → {pct(o1)} · KL(π₀‖π) {num(kl, 2)}")
    stored = RAZOR["live"]["runs"][mth]
    assert abs(o1 - stored["old"][-1]) < 0.02 and abs(kl - stored["kl"][-1]) < 0.05 * max(1, stored["kl"][-1])

SFT forgets almost all of Fashion-MNIST, RL forgets part of it, optimal SFT forgets almost nothing. And the KL follows
exactly the same order. The full version: 3 methods × 3 learning rates × 3 seeds, measured throughout training (the
number of steps is printed below). For a fair comparison, we take the measurements at which the new task has reached
88–90%:

In [ ]:
pts = {m: [] for m in ("sft", "rl", "oracle")}
for x in RAZOR["runs"]:
    base_old = x["old"][0]
    pts[x["method"]] += [(k, base_old - o, n, x["lr"]) for n, o, k in zip(x["new"][1:], x["old"][1:], x["kl"][1:])]
for m, mp in pts.items():
    sel = [p for p in mp if 0.88 <= p[2] <= 0.90]
    f = np.array([p[1] for p in sel])
    print(f"{names[m]:23} {len(sel):3} measurements · median forgetting {pct(np.median(f))} (from {pct(f.min())} to {pct(f.max())}) "
          f"· median KL {num(np.median([p[0] for p in sel]), 2)}")  # fmt: skip
print(f"({num(RAZOR['steps'], 0)} steps per run, measured every {RAZOR['every']} steps; the new task reaches at most "
      f"{pct(max(max(x['new']) for x in RAZOR['runs']))})")  # fmt: skip
for x_lr in sorted({x["lr"] for x in RAZOR["runs"]}):
    med = {m: np.median([p[1] for p in mp if p[3] == x_lr and 0.88 <= p[2] <= 0.90]) for m, mp in pts.items()}
    print(f"η {num_lr(x_lr)}: median forgetting (measurements at 88–90%) SFT {num(100 * med['sft'], 1)} · "
          f"RL {num(100 * med['rl'], 1)} · optimal SFT {num(100 * med['oracle'], 1)} points")  # fmt: skip
    assert med["sft"] > med["rl"] > med["oracle"], "the forgetting order differs at this learning rate"
print(
    f"\nforgetting vs. KL, quadratic fit over {sum(map(len, pts.values()))} measurements: R² = {num(RAZOR['fit']['r2'], 3)}"
)


def r2_quad(x, y):
    A = np.stack([np.ones_like(x), x, x * x], 1)
    c, *_ = np.linalg.lstsq(A, y, rcond=None)
    e = y - A @ c
    return 1 - (e @ e) / ((y - y.mean()) @ (y - y.mean()))


c0, c1, c2 = RAZOR["fit"]["coef"]
for x_lr in sorted({x["lr"] for x in RAZOR["runs"]}):
    res = [
        (x["old"][0] - o) - (c0 + c1 * k + c2 * k * k)
        for x in RAZOR["runs"]
        if x["method"] == "rl" and x["lr"] == x_lr
        for k, o in zip(x["kl"][1:], x["old"][1:])
    ]
    print(f"RL, η {num_lr(x_lr)}: on average {num(100 * abs(np.mean(res)), 1)} points "
          f"{'above' if np.mean(res) > 0 else 'below'} the common curve")  # fmt: skip
k_old = np.array([k for x in RAZOR["runs"] for k in x["klOld"][1:]])
forget = np.array([x["old"][0] - o for x in RAZOR["runs"] for o in x["old"][1:]])
print(f"the same fit, but with the KL measured on the Fashion-MNIST images: R² = {num(r2_quad(k_old, forget), 3)}")

fig, ax = plt.subplots(figsize=(7, 4))
for m, col, mk in (("sft", ORANGE, "s"), ("rl", BLUE, "o"), ("oracle", GREEN, "^")):
    k = np.array([p[0] for p in pts[m]])
    f = np.array([p[1] for p in pts[m]])
    ax.scatter(np.maximum(k, 1e-3), f, s=10, color=col, marker=mk, alpha=0.5, label=names[m])
kl_all = [p[0] for mp in pts.values() for p in mp]
xs = np.logspace(math.log10(max(min(kl_all), 1e-3)), math.log10(max(kl_all)), 200)  # only over the measured KL range
c0, c1, c2 = RAZOR["fit"]["coef"]
ax.plot(xs, c0 + c1 * xs + c2 * xs**2, color="black", lw=1, label="quadratic fit")
ax.set(xscale="log", xlabel="KL(π₀ ‖ π) on the parity images", ylabel="Fashion-MNIST forgetting (percentage points)")
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: num(100 * v, 0)))
ax.legend(frameon=False, fontsize=8)
fmt_axes(fig)
plt.show()

Each point is one measurement (one method, one learning rate, one seed, one step): the horizontal axis is the KL from
the original model (log scale), the vertical axis is the drop on Fashion-MNIST. SFT (orange, squares) sits on the
right and high up, optimal SFT (green, triangles) on the left and low down, RL (blue, circles) in between — and all
three lie **close to** one common curve. The forgetting depends mainly on how far the model has moved away from
$\pi_0$, rather than on "is it RL or SFT". RL forgets less because it samples from itself: it is rewarded only for
answers it already tended to pick, so it moves toward a correct answer close to $\pi_0$ without anyone pointing it
out.

Two places where the curve doesn't fit perfectly (printed in the cell above): RL with the largest learning rate sits
about 10 points **above** the common curve, i.e. it forgets more than its KL predicts; and the KL measured on the
Fashion-MNIST images themselves predicts the forgetting almost as well. So "a single KL number" is a good summary here,
not yet a law.

**Conclusion: holds, with one caveat.** The forgetting order SFT > RL > optimal SFT holds at all three learning rates
(printed in the cell above), and the KL from the original model predicts most of the forgetting ($R^2 = 0.93$; the
paper: 0.96). But at the largest learning rate, RL forgets only slightly less than SFT (59.2 against 61.1 points) and
departs from the common curve. Limitations: one toy problem, one architecture; and the new task
doesn't go beyond the level printed above, because this small model only recognizes digits that well.

## 7 · Chain of thought on the parity problem

**Claim** (Kim & Suzuki, ICLR 2025): learning **sparse parity** directly — the XOR of $k$ bits at secret positions
among $n$ bits — is hard for gradient methods: the loss stays flat at chance level for a very long time. If the model
is taught to write out the intermediate results (the parity of the first 1, 2, …, $k$ secret bits, one token per
result), each step is just the XOR of 2 bits, and a small transformer learns it very quickly. They prove this for a
one-layer transformer.

The experiment: $n = 20$ bits, $k = 4$ fixed secret positions, fresh random inputs at every step (so there is no
training set to memorize). A causal transformer with 2 blocks, $d = 64$, 4 heads.

In [ ]:
SECRET = torch.tensor(COT["tasks"]["n20k4"]["secret"])
N_BITS = COT["tasks"]["n20k4"]["n"]


class Tiny(nn.Module):
    def __init__(self, T, d=64):
        super().__init__()
        self.emb = nn.Embedding(2, d)
        self.pos = nn.Parameter(torch.randn(T, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 4, 2 * d, dropout=0.0, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.head = nn.Linear(d, 2)
        self.register_buffer("mask", torch.triu(torch.ones(T, T, dtype=torch.bool), 1))

    def forward(self, x):
        L = x.shape[1]
        return self.head(self.enc(self.emb(x) + self.pos[:L], mask=self.mask[:L, :L], is_causal=True))


def parity_run(cot, steps, seed=0, every=100):
    """Exactly as in reel/research/cot.py."""
    n, S, k = N_BITS, SECRET, len(SECRET)
    torch.manual_seed(seed)
    m = Tiny(n + k)
    opt = torch.optim.AdamW(m.parameters(), 1e-3)
    g, ge = torch.Generator().manual_seed(1000 + seed), torch.Generator().manual_seed(2000 + seed)
    acc = []
    for s in range(1, steps + 1):
        x = torch.randint(0, 2, (256, n), generator=g)
        bits = x[:, S]
        tgt = torch.cumsum(bits, 1) % 2 if cot else bits.sum(1, keepdim=True) % 2
        inp = torch.cat([x, tgt[:, :-1]], 1) if cot else x
        loss = nn.functional.cross_entropy(m(inp)[:, n - 1 :].reshape(-1, 2), tgt.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        if s % every == 0:
            with torch.no_grad():
                x = torch.randint(0, 2, (2000, n), generator=ge)
                ans = x[:, S].sum(1) % 2
                if cot:  # generate k tokens one after another; the last token is the answer
                    inp = x
                    for _ in range(k):
                        inp = torch.cat([inp, m(inp)[:, -1].argmax(1, keepdim=True)], 1)
                    pred = inp[:, -1]
                else:
                    pred = m(x)[:, -1].argmax(1)
                acc.append((pred == ans).float().mean().item())
    return acc


print(f"{N_BITS} bits, secret positions {SECRET.tolist()}")

### 🤔 Predict first
Learning the answer directly, what percentage does the model get right after 300 steps? And if it writes 4
intermediate tokens before answering? (The cell below runs only the chain-of-thought version live; the direct version
takes the same time and gets nowhere, so we read its results, same seed, from the saved file.)

In [ ]:
cot_live = parity_run(True, 300, every=300)  # measure only at the last step, to save time
saved = {k: COT["runs"][k][0]["acc"][:3] for k in ("n20k4-direct", "n20k4-cot")}  # seed 0, both ways
print(
    "direct answer    (saved): accuracy after 100 / 200 / 300 steps = "
    + " / ".join(pct(a) for a in saved["n20k4-direct"])
)
print(
    "chain of thought (saved): accuracy after 100 / 200 / 300 steps = " + " / ".join(pct(a) for a in saved["n20k4-cot"])
)
print(f"chain of thought, live run: after 300 steps = {pct(cot_live[-1])}")
assert abs(cot_live[-1] - saved["n20k4-cot"][2]) < 0.02, "differs from the saved results"

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for key, col, ls, label in (("n20k4-direct", ORANGE, "--", "n 20, k 4 · direct answer"),
                            ("n20k4-cot", BLUE, "-", "n 20, k 4 · chain of thought"),
                            ("n24k6-direct", GREY, ":", "n 24, k 6 · direct answer"),
                            ("n24k6-cot", GREEN, "-.", "n 24, k 6 · chain of thought")):  # fmt: skip
    for i, r_ in enumerate(COT["runs"][key]):
        steps = np.arange(1, len(r_["acc"]) + 1) * COT["eval"]
        ax.plot(steps, r_["acc"], color=col, ls=ls, lw=1.4, label=label if i == 0 else None)
ax.set(xscale="log", xlabel="training step", ylabel="accuracy (2,000 fresh inputs)", ylim=(0.4, 1.02))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: pct(v, 0)))
ax.legend(frameon=False, fontsize=8)
fmt_axes(fig)
plt.show()
for key in COT["runs"]:
    rs = COT["runs"][key]
    print(
        f"{key:13}: steps to reach 99%: {steps_text([r_['solved'] for r_ in rs])} · final {', '.join(pct(r_['acc'][-1]) for r_ in rs)}"
    )

Each line is one seed (3 seeds per configuration); the horizontal axis is the number of steps (log scale). Direct
learning (orange and gray, dashed and dotted) stays at 50% for all 10,000 steps, in all 3 seeds and on both problems:
the model doesn't find the secret positions. With chain of thought (blue and green), it solves the problem within 300
steps for $k = 4$, and in 500–700 steps for $k = 6$.

**Conclusion: holds.** Writing out the intermediate steps turns a problem in which the gradient can find almost no
way in into a chain of easy problems. Limitation: here the chain of thought is **taught** (teacher forcing on the
correct steps); the paper also considers this case. The model doesn't come up with how to break the problem down by
itself.

## Summary
- **Copying or creating — the main part holds.** The exact denoiser only copies (100%); restricted to local patches,
  it makes no copies at all for $P \le 9$, and patches of size 5–9 produce new handwriting. The quality depends
  strongly on the patch size.
- **When copying starts — holds.** The onset of copying grows roughly in proportion to the amount of data (500 steps
  with 32 codes, 30,000–40,000 with 1,024), while sample quality is reached early and stays almost unchanged.
- **Model collapse — holds on both sides.** Replacing the real data with synthetic data collapses; keeping the old
  data doesn't. The two papers describe two different procedures; they don't contradict each other.
- **RL's Razor — holds.** SFT forgets the most, RL less, optimal SFT the least, and the forgetting of all three lies on
  one curve of $\mathrm{KL}(\pi_0 \| \pi)$ ($R^2 = 0.93$).
- **Chain of thought — holds.** Learning the parity of 4 or 6 secret bits directly stays at chance level for all
  10,000 steps; writing out the intermediate steps solves it in a few hundred steps.

## Review questions
1. Why can the exact denoiser only return training images?
2. Why do patches that are too small give fragments, while patches that are too large come close to copying?
3. In the normal-distribution experiment, each estimation step is only slightly off. So why does the standard
   deviation still shrink steadily when the data is replaced, even when dividing by $n - 1$ for an unbiased variance
   estimate?
4. Why is a parity task with "5 correct answers" the condition for RL and SFT to forget differently?
5. Nobody shows RL the answer closest to $\pi_0$. Why does it still end up closer to that answer than SFT does?
6. Why is learning sparse parity directly hard for gradients, while each step of a two-bit XOR is easy?

<details><summary>Hints and answers</summary>

1. It is a weighted average of the training images, and as $t \to 0$ all the weight goes to the nearest image.
2. With small patches, each pixel picks its own training image and nothing holds the strokes together; with large
   patches, neighboring pixels see almost the same region, so they pick the same image, and the solution approaches
   the ideal one.
3. Each generation multiplies σ by a random factor around 1, and a decrease is not fully made up for by an increase of
   the same size, so the product of the factors drifts downward, with no real data to pull it back. `pool.std()`
   (dividing by $n$) adds a small downward bias each time as well; removing that bias (dividing by $n - 1$) only makes
   σ shrink more slowly.
4. When there is only one correct answer, every training method has to arrive at the same distribution; when there
   are many correct answers, each method arrives at a different distribution, and they lie at different distances
   from $\pi_0$.
5. It learns from its own samples, so it is rewarded only for the correct answers it already tended to pick, and it
   pushes up their probabilities in that same proportion.
6. Until all $k$ bits are present, every combination that is missing one bit is uncorrelated with the answer, so the
   gradient is almost 0. Each chain-of-thought step only has to find **one** secret position: the first step copies a
   bit, so the bit in the right position is directly correlated with the answer; each later step is the XOR of the
   token at the very position making the prediction (the previous step's result, already in the input) with one new
   bit, so the model only has to search $n$ positions instead of every combination of $k$ positions at once.
</details>

## Further reading
- M. Kamb & S. Ganguli (2024), [An analytic theory of creativity in convolutional diffusion
  models](https://arxiv.org/abs/2412.20292); A. Lukoianov et al. (2025), [Locality in Image Diffusion Models Emerges
  from Data Statistics](https://arxiv.org/abs/2509.09672).
- T. Bonnaire, R. Urfin, G. Biroli & M. Mézard (2025), [Why Diffusion Models Don't Memorize: The Role of Implicit
  Dynamical Regularization in Training](https://arxiv.org/abs/2505.17638).
- I. Shumailov et al. (2023), [The Curse of Recursion: Training on Generated Data Makes Models
  Forget](https://arxiv.org/abs/2305.17493); M. Gerstgrasser et al. (2024), [Is Model Collapse Inevitable? Breaking
  the Curse of Recursion by Accumulating Real and Synthetic Data](https://arxiv.org/abs/2404.01413).
- I. Shenfeld, J. Pari & P. Agrawal (2025), [RL's Razor: Why Online Reinforcement Learning Forgets
  Less](https://arxiv.org/abs/2509.04259).
- J. Kim & T. Suzuki (2024), [Transformers Provably Solve Parity Efficiently with Chain of
  Thought](https://arxiv.org/abs/2410.08633).

**Back to:** [Lesson 26 · From language model to assistant](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/hau-huan-luyen/).

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · torch {torch.__version__} · numpy {np.__version__} · "
      f"matplotlib {matplotlib.__version__}")  # fmt: skip